# SAM3.1 MLX Autoresearch — Analysis

Visualize experiment results: accuracy vs speed Pareto frontier.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("results.tsv", sep="\t")
print(f"Total experiments: {len(df)}")
print(f"Kept: {(df.status == 'keep').sum()}")
print(f"Discarded: {(df.status == 'discard').sum()}")
print(f"Crashed: {(df.status == 'crash').sum()}")
print(f"Keep rate: {(df.status == 'keep').sum() / max(1, (df.status != 'crash').sum()):.1%}")

In [ ]:
# List all kept experiments
kept = df[df.status == 'keep'].reset_index()
for _, row in kept.iterrows():
    print(f"  #{row['index']:3d} | {row.ms_per_frame:6.1f} ms | IoU={row.mean_iou:.4f} | {row.memory_gb:.1f}GB | {row.description}")

In [ ]:
# Progress plot: ms_per_frame over experiments
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Top: ms_per_frame
colors = {'keep': '#2ecc71', 'discard': '#95a5a6', 'crash': '#e74c3c'}
for status, color in colors.items():
    mask = df.status == status
    ax1.scatter(df.index[mask], df.ms_per_frame[mask], c=color, s=40, alpha=0.7, label=status, zorder=3)

# Running best (minimum ms_per_frame among kept)
kept_mask = df.status == 'keep'
if kept_mask.any():
    running_best = df.ms_per_frame.where(kept_mask).ffill()
    running_best = running_best.cummin()
    ax1.step(df.index, running_best, where='post', color='#27ae60', linewidth=2, label='best', zorder=4)

ax1.set_ylabel('ms / frame (lower is better)')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_title('SAM3.1 Inference Speed Optimization')

# Bottom: mean_iou
for status, color in colors.items():
    mask = df.status == status
    ax2.scatter(df.index[mask], df.mean_iou[mask], c=color, s=40, alpha=0.7, zorder=3)

ax2.axhline(y=0.99, color='red', linestyle='--', alpha=0.5, label='99% threshold')
ax2.set_ylabel('Mean IoU (higher is better)')
ax2.set_xlabel('Experiment #')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('progress.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved progress.png")

In [ ]:
# Pareto frontier: accuracy vs speed
valid = df[df.status != 'crash'].copy()
if len(valid) > 0:
    fig, ax = plt.subplots(figsize=(10, 7))
    
    for status, color in colors.items():
        mask = valid.status == status
        ax.scatter(valid.ms_per_frame[mask], valid.mean_iou[mask], c=color, s=60, alpha=0.7, label=status)
    
    # Pareto frontier
    kept_only = valid[valid.status == 'keep'].sort_values('ms_per_frame')
    if len(kept_only) > 0:
        pareto_ms = [kept_only.iloc[0].ms_per_frame]
        pareto_iou = [kept_only.iloc[0].mean_iou]
        for _, row in kept_only.iterrows():
            if row.mean_iou >= pareto_iou[-1]:
                pareto_ms.append(row.ms_per_frame)
                pareto_iou.append(row.mean_iou)
        ax.plot(pareto_ms, pareto_iou, 'g--', linewidth=2, alpha=0.5, label='Pareto frontier')
    
    ax.axhline(y=0.99, color='red', linestyle='--', alpha=0.3)
    ax.set_xlabel('ms / frame')
    ax.set_ylabel('Mean IoU')
    ax.set_title('Accuracy vs Speed Tradeoff')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('pareto.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Summary statistics
if len(kept) > 0:
    baseline_ms = kept.iloc[0].ms_per_frame
    best_ms = kept.ms_per_frame.min()
    best_iou = kept[kept.ms_per_frame == best_ms].mean_iou.iloc[0]
    speedup = baseline_ms / best_ms
    
    print(f"Baseline:     {baseline_ms:.1f} ms/frame")
    print(f"Best:         {best_ms:.1f} ms/frame (IoU={best_iou:.4f})")
    print(f"Speedup:      {speedup:.2f}x")
    print(f"Improvement:  {baseline_ms - best_ms:.1f} ms saved per frame")
    print(f"\nTop improvements:")
    
    # Show biggest jumps
    prev_best = baseline_ms
    for _, row in kept.iterrows():
        if row.ms_per_frame < prev_best:
            delta = prev_best - row.ms_per_frame
            print(f"  -{delta:.1f}ms: {row.description}")
            prev_best = row.ms_per_frame